In [78]:
import pandas as pd
from snappy import Link
df = pd.read_excel("./knotinfo_data_complete.xls")
filtered = df[(df["unknotting_number"].astype(str) == "[2,3]") | (df["unknotting_number"].astype(str) == "[2,4]")  & ((df["signature"].astype(str) == "4") | (df["signature"].astype(str) == "-4"))]
all_filtered = filtered[filtered["alternating"].str.contains("Y")][["name", "two_bridge_notation", "signature", "unknotting_number"]]
all_Knot_Names = all_filtered["name"].to_list()
print(all_filtered)

          name two_bridge_notation signature unknotting_number
91        10_6              [37,7]        -4             [2,3]
96       10_11             [43,10]        -2             [2,3]
132      10_47                 NaN         4             [2,3]
136      10_51                 NaN         2             [2,3]
139      10_54                 NaN         2             [2,3]
...        ...                 ...       ...               ...
7834  13a_4856             [47,39]        -2             [2,3]
7839  13a_4861                 NaN        -4             [2,4]
7847  13a_4869                 NaN        -4             [2,4]
7849  13a_4871                 NaN        -4             [2,4]
7850  13a_4872             [85,69]        -4             [2,4]

[1290 rows x 4 columns]


In [79]:
import itertools
from snappy import Link

All_Qs = dict()
Knot_Names = dict()

def load_pickle_file(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

def dump_pickle_file(file_path, file_data):
    with open(file_path,"wb") as file:
        pickle.dump(file_data, file)
    return True
    
def d_invariant_of_two_bridged_knots(p, q):
    def d(p, q, i):
        return 0 if q == 0 else 1/4 - 1/4 * (2 * i + 1 - p - q)^2 / (p*q) - d(q, p % q, i %q)
    original_dlist = [d(p, q, i) for i in range(0, p)]
    # Rotate the resulting list to align with the cyclic group structure
    k = (q-1)/2 if q%2 == 1 else (p+q-1)/2
    dlist = vector(original_dlist[k:] + original_dlist[:k])
    return dlist if dlist[0] < 0 else -1 * dlist

def convert_lattice_to_group_word(lattice_element, group):
    gens = group.gens()
    group_element = gens[0]^lattice_element[0]
    for i in range(1, len(lattice_element)):
        group_element *= gens[i]^lattice_element[i]
    return group_element

def quotient_by_lattice(Q):
    r = rank(Q)
    G = FreeGroup(rank(Q))
    gens = G.gens()
    commutators = [gens[i]*gens[j]*gens[i]^(-1)*gens[j]^(-1) for i in range(r-1) for j in range(i+1, r)]
    lattice_relation_vectors = (Q).columns()
    return G.quotient(commutators + [convert_lattice_to_group_word(x,G) for x in lattice_relation_vectors])

def get_characteristic_covectors(Q):
    each_component_options = [list(range(-Q[i][i], Q[i][i]-1, 2)) for i in range(rank(Q))]
    return itertools.product(*each_component_options)
                   
def make_intersection_form(m2, m1, a):
    return matrix([[m1, 1, a, 0], [1, 2, 0, 0], 
                   [a, 0, m2, 1], [0, 0, 1, 2]])

def find_all_possible_intersection_forms(detK, n = 2):
    possibleQ = []
    for a in range(0, detK//4 + 1):
        for k1 in divisors(detK + 4 * a^2):
            k2 = (detK + 4 * a^2)/k1
            if k1 <= k2 and sum([x%4 for x in [k1,k2]]) == 3 * n + 1 * (2-n):
                m1 = (k1 + 1)/2
                m2 = (k2 + 1)/2
                if a < m1:
                    possibleQ += [make_intersection_form(m2, m1, a)]
    return possibleQ

def get_group_element(covector, Q):
    group = quotient_by_lattice(Q)
    return convert_lattice_to_group_word(covector, group)

def get_dlist_Q(Q):
    unique_char_cvectors = []
    generators = []
    for char_covector in get_characteristic_covectors(Q):
        m_covector = matrix(char_covector)
        current_d_value = (m_covector * Q^(-1) * m_covector.transpose() - rank(Q))/4
        current_group_element = get_group_element(char_covector, Q)
        if current_group_element.order() == det(Q):
            generators += [current_group_element]
        unique = True
        for i in range(len(unique_char_cvectors)):
            (group_element, d_value) = unique_char_cvectors[i]
            if current_group_element == group_element:
                unique = False
                unique_char_cvectors[i] = (group_element, min([d_value, current_d_value]))
                break
        if unique:
            unique_char_cvectors += [(current_group_element, current_d_value)]
    new_dlist = []
    if len(generators) > 0:
        generator = generators[0]
        for i in range(det(Q)):
            current_group_element = generator^i
            for (group_element, d_value) in unique_char_cvectors:
                if group_element == current_group_element:
                    new_dlist += [d_value[0][0]]
                    break
    return new_dlist

def pretty_print(original_dlist):
    new_list = []
    length = len(original_dlist)
    for i in range(length//15 + 1):
        new_list += [[]]
        for j in range(15):
            pointer = i*15 + j
            if pointer >= length:
                new_list[i] += [0]
            else:
                new_list[i] += [original_dlist[pointer]]
    print(matrix(new_list))

def obstruct(dlist, Q):
    # Note if the group is cyclic we may assume an automorphism of Z_p
    Q_dlist = get_dlist_Q(Q)
    p = Integer(det(Q))
    if len(Q_dlist) != p:
        print ("Obstructed by Z^n/Q not being cyclic")
        return True
  
    (shift_for_simplicity, _) = find_minimal_generator(Q_dlist)
    isomorphic_Q_dlist = vector(isomorphic_dlist(Q_dlist, shift_for_simplicity))
    pretty_print(isomorphic_Q_dlist)
    
    for possible_isomorphism_shift in p.coprime_integers((p+1)/2):
        isomorphism_attempt_d_values = []
        isomorphism = vector(isomorphic_dlist(dlist, possible_isomorphism_shift))
        for group_index in range(1, p):
            isomorphism_attempt_d_values += [isomorphism[group_index]]
            difference_in_values = isomorphism[group_index] - isomorphic_Q_dlist[group_index]
            obs_diff_mod_2 = not (difference_in_values / 2).is_integer()
            if isomorphism[group_index] > isomorphic_Q_dlist[group_index] or obs_diff_mod_2:
                break
        if len(isomorphism_attempt_d_values) < p - 1:
            if len(isomorphism_attempt_d_values) > 1:
                print(list(isomorphic_Q_dlist[1: len(isomorphism_attempt_d_values) + 1]))
                print(possible_isomorphism_shift, isomorphism_attempt_d_values)
        else:
            print("Unobstructed")
            print(possible_isomorphism_shift)
            return False
    return True

def find_minimal_generator(MCQ_list):
    p = Integer(len(MCQ_list))
    gens = [(i, MCQ_list[i]) for i in p.coprime_integers(p)]
    return min(gens, key = lambda generator: generator[1])

def isomorphic_dlist(dlist, shift):
    p = len(dlist)
    assert gcd(shift, p) == 1
    return [dlist[i*shift % p] for i in range(p)]

def obstruct_alternating(name, expected_signature = 4):
    print(f"{name}")
    L = Link(name)
    assert L.signature() in [-expected_signature, expected_signature]
    assert L.is_alternating()
    signature = abs(L.signature())
    G = L.goeritz_matrix()
    G = G if G.is_positive_definite() else L.mirror().goeritz_matrix()
    assert G.is_positive_definite()
    detK = abs(det(G))
    dlist = vector(get_dlist_Q(G))
    if len(dlist) != detK:
        print("H_1 not cyclic will skip for now")
        return
    if not dlist[0] in [-signature/4, signature/4]:
        print("Bad Goeritz matrix. Should check")
        return
    dlist = -dlist if dlist[0] == signature/4 else dlist # correct possible sign error 
    print(f"Correction terms")
    pretty_print(dlist)
    if detK in All_Qs:
        QS = All_Qs[detK]
    else:
        QS = find_all_possible_intersection_forms(detK, n = signature/2)
    All_Qs[detK] = QS
    unobstructed = False
    for Q in QS:
        print("Testing obstruction for")
        print(f"m1: {Q[0][0]}, m2: {Q[2][2]},a: {Q[0][2]}")
        unobstructed = not obstruct(dlist, Q)
        if unobstructed:
            break
    print("---------------------------")
    if not unobstructed:
        print(f"SUCCESS for name = {name}")
        Knot_Names[name] = True
    print("---------------------------")

def obstruct_twobridge(p,q, name):
    print(f"Obstructing p = {p} | q = {q}")
    dlist = d_invariant_of_two_bridged_knots(p,q)
    print(f"Correction terms")
    pretty_print(dlist)
    if p in All_Qs:
        QS = All_Qs[p]
    else:
        QS = find_all_possible_intersection_forms(p)
    All_Qs[p] = QS
    unobstructed = False
    for Q in QS:
        print("Testing obstruction for")
        print(f"m1: {Q[0][0]}, m2: {Q[2][2]},a: {Q[0][2]}")
        unobstructed = not obstruct(dlist, Q)
        if unobstructed:
            break
    print("---------------------------")
    if not unobstructed:
        print(f"SUCCESS for p = {p} | q = {q} | name = {name}")
        Knot_Names[name] = True
    print("---------------------------")

In [74]:
obstruct_alternating("12a150", expected_signature=2)

12a150
Correction terms
[    -1/2  -19/318 -235/318  155/106  173/318  161/318  -69/106  341/318 -103/318  123/106 -151/318  245/318 -117/106  -31/318  -67/318]
[  59/106   65/318  233/318   15/106  137/318 -127/318  175/106  185/318  125/318  -97/106  209/318 -283/318   47/106  209/318  -79/318]
[ -29/106  185/318  101/318   99/106  137/318  257/318    7/106   65/318 -247/318  119/106  -31/318 -139/318   11/106 -151/318  -55/318]
[ 107/106  341/318    5/318  -17/106  173/318   41/318   63/106  -19/318      1/6  -77/106 -235/318   41/318  -13/106  161/318    5/318]
[  43/106 -103/318  -55/318   91/106  245/318 -139/318  -81/106  -67/318 -247/318  -49/106  233/318  257/318  -25/106 -127/318  101/318]
[  -9/106  125/318  -79/318   -1/106 -283/318 -283/318   -1/106  -79/318  125/318   -9/106  101/318 -127/318  -25/106  257/318  233/318]
[ -49/106 -247/318  -67/318  -81/106 -139/318  245/318   91/106  -55/318 -103/318   43/106    5/318  161/318  -13/106   41/318 -235/318]
[ -77/106      1/

In [75]:
obstruct_alternating("13a4873", expected_signature=2)

13a4873
Correction terms
[   -1/2  49/110 141/110   1/110  69/110   25/22 169/110 201/110   1/110   9/110   45/22   -1/10 -39/110 141/110  89/110]
[   5/22 169/110  81/110 201/110  89/110   -7/22  49/110   11/10 -39/110   9/110   53/22  69/110  81/110  81/110  69/110]
[  53/22   9/110 -39/110   11/10  49/110   -7/22  89/110 201/110  81/110 169/110    5/22  89/110 141/110 -39/110   -1/10]
[  45/22   9/110   1/110 201/110 169/110   25/22  69/110   1/110 141/110  49/110       0       0       0       0       0]
Testing obstruction for
m1: 1, m2: 28,a: 0
[   -1/2 -51/110 -39/110 -19/110   9/110    9/22  89/110 141/110 201/110 269/110   69/22   39/10 521/110 621/110 729/110]
[ 125/22 529/110 441/110 361/110 289/110   45/22 169/110   11/10  81/110  49/110    5/22   9/110   1/110   1/110   9/110]
[   5/22  49/110  81/110   11/10 169/110   45/22 289/110 361/110 441/110 529/110  125/22 729/110 621/110 521/110   39/10]
[  69/22 269/110 201/110 141/110  89/110    9/22   9/110 -19/110 -39/110 -51/1